# Data Cleansing

## Import all requirements

In [344]:
import pandas as pd
import numpy as np
import missingno as msgn
from narwhals import col
from numpy import dtype
from prometheus_client.utils import NaN

url = '../data/airbnb.csv'
airbnb = pd.read_csv(url)

## Price fixing
### Remove dollar sign

In [345]:
# airbnb[~airbnb['price'].str.contains('$', regex=False, na=False)]
# airbnb['price'] = airbnb['price'].str.slice(0,-1)
airbnb['price'] = airbnb['price'].str.replace('$', '', regex=False)

### Convert prices to numeric values FLOAT

In [346]:
airbnb['price'] = pd.to_numeric(airbnb['price'])
airbnb['price'] = airbnb['price'].fillna(airbnb['price'].median())

In [347]:
airbnb.rename(columns={'Unnamed: 0' : 'ID'}, inplace=True)

## Coordinates cleaning

In [348]:
coords = airbnb['coordinates'].str.slice(1, -1).str.partition(',').drop(columns=1)
airbnb['longitude'] = coords[0]
airbnb['latitude'] = coords[2]
airbnb.drop(columns='coordinates', inplace=True)
airbnb['room_type'].unique()

<StringArray>
[        'Private room',      'Entire home/apt',              'Private',
          'Shared room',         'PRIVATE ROOM',                 'home',
 '   Shared room      ']
Length: 7, dtype: str

## Cleaning of the room types

In [349]:
airbnb['room_type'].unique()
# airbnb[airbnb['room_type'] == 'home']['room_type'] = 'Entire home/apt'
# airbnb[airbnb['room_type'] == 'home']['room_type'] = 'Entire home/apt'
for x in airbnb.index:
    if airbnb.loc[x, 'room_type'] == 'home':
        airbnb.loc[x, 'room_type'] = 'Entire home/apt'
    elif airbnb.loc[x, 'room_type'] == 'Private' or airbnb.loc[x, 'room_type'] == 'PRIVATE ROOM':
        airbnb.loc[x, 'room_type'] = 'Private room'
    elif airbnb.loc[x, 'room_type'] == '   Shared room      ':
        airbnb.loc[x, 'room_type'] = 'Shared room'

airbnb['room_type'].unique()

<StringArray>
['Private room', 'Entire home/apt', 'Shared room']
Length: 3, dtype: str

In [350]:
airbnb.isna().any()
# we will use mode to replace nan values

ID                    False
listing_id            False
name                   True
host_id               False
host_name              True
neighbourhood_full    False
room_type             False
price                 False
number_of_reviews     False
last_review            True
reviews_per_month      True
availability_365      False
rating                 True
number_of_stays        True
5_stars                True
listing_added         False
longitude             False
latitude              False
dtype: bool